# Transformación de género musical

## Descripción del caso

Un usuario aporta una canción y un género objetivo. La IA generativa prepara una versión nueva con la misma estructura y en el género pedido. La pieza tiene que seguir siendo reconocible: se mantienen la melodía principal, la estructura general y la letra, si la canción original la tiene. Cambia lo propio del género de partida: instrumentación, ritmo y percusión, técnicas de interpretación, dinámica, texturas y producción.

El caso con el que se validó el prompt es `jazz.00004.wav` transformado a clásico. El prompt pide tres cosas a la vez, y en este orden: reconocibilidad de la canción, autenticidad del género objetivo y coherencia musical.

En las pruebas se comparó la misma tarea con ChatGPT, Mureka y Gemini.

| Herramienta | Prueba | Resultado |
|---|---|---|
| ChatGPT | Metal → clásico | Sirvió para redactar el prompt. No generó el WAV. |
| Mureka | Metal → clásico | Mezcló clásico con pop e inventó una letra. |
| Mureka | Metal → clásico, prompt más corto | Más presencia del género objetivo, menos semejanza con la original y sin letra. |
| Mureka | Pop → jazz | Jazz reconocible y bastante semejante, sin llegar a ser del todo la misma canción. |
| Gemini | Jazz → clásico | Casi la misma armonía y género clásico claro. Mejor resultado de las pruebas. |

Por eso este notebook usa el system prompt y el user prompt de la prueba con Gemini, sin reescribirlos. Quien lo ejecuta sube su WAV, el nombre del archivo entra en el prompt y el modelo devuelve la respuesta escrita de la transformación.


## 1. Configuración de la API

In [ ]:
# pip install -U -q "google-genai"
from google import genai
from google.genai.types import GenerateContentConfig

GOOGLE_API_KEY = ""  # Pega aquí tu API Key de Google AI Studio
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)


## 2. Carga de la canción

La canción la aporta quien ejecuta la prueba. En Colab, `files.upload()` abre el selector y el nombre del WAV queda en `cancion`. Ese nombre es el que el prompt cita como canción original.

No se transcribe la letra ni se calculan tempo, centroide espectral ni otras features antes de la llamada. El prompt ya le pide al modelo que identifique los elementos del género original y los sustituya por los del género objetivo. Un resumen previo sustituiría esa instrucción y no forma parte del prompt que ganó las pruebas.


In [ ]:
from google.colab import files

uploaded = files.upload()
cancion = list(uploaded.keys())[0]
print("Canción:", cancion)


## 3. Configuración del modelo y llamada

`cancion` es el archivo subido en el apartado anterior. El género objetivo de esta prueba es clásico.

| Parámetro | Valor | Motivo |
|---|---|---|
| model | gemini-2.5-flash | Modelo de las prácticas con system_instruction. Sigue instrucciones cerradas y alcanza para esta prueba. |
| temperature | 0.8 | La tarea es un arreglo, así que hace falta algo de variación. 0.8 deja margen para la instrumentación del género objetivo sin aflojar las restricciones de melodía, estructura y letra. |
| system_instruction | system_prompt | Fija el rol de productor, arreglista e ingeniero de sonido, y las dos restricciones de toda la tarea: no inventar géneros y no convertir la canción en otra composición. |
| contents | prompt | Encargo concreto: canción, género objetivo, criterios de transformación y restricciones. Es el texto validado en las pruebas. |
| Género en el prompt | `clásico` | Género de la prueba en la que este prompt quedó por delante de las otras formulaciones. |



In [ ]:
system_prompt = """Eres un experto productor musical, arreglista e ingeniero de sonido.
Tu tarea es transformar la canción proporcionada al género musical objetivo indicado.
No inventes géneros adicionales. No conviertas la canción en una composición distinta."""

prompt = f"""Transforma la canción original al género objetivo.

CANCIÓN ORIGINAL:
```{cancion}```

GÉNERO OBJETIVO:
```clásico```

Para transformar la canción original en el género objetivo sigue estos criterios:

1. Mantén siempre la melodía principal y la estructura general.

2. Identifica qué elementos pertenecen al género original y sustitúyelos o modifícalos por los elementos del género objetivo:
- instrumentación característica del género objetivo
- ritmo y percusión propios del género objetivo
- técnicas de interpretación del estilo solicitado
- dinámica
- texturas y producción
- elementos característicos del género objetivo

3. Siempre que sea posible, mantén la letra original. Si la canción original no tiene letra, la canción generada debe ir también sin letra. No inventes una letra.

RESTRICCIONES:
- No hagas una copia de la producción original.
- No conviertas la canción en una composición completamente diferente.
- No introduzcas géneros no solicitados, salvo elementos secundarios naturalmente asociados al género objetivo.
- Prioriza siempre: reconocibilidad de la canción + autenticidad del género objetivo + coherencia musical."""

model = "gemini-2.5-flash"
temperature = 0.8

response = gemini_client.models.generate_content(
    model=model,
    contents=prompt,
    config=GenerateContentConfig(
        system_instruction=system_prompt,
        temperature=temperature
    )
)

print(response.text)
